# Backtest Statistics


Strategy: **sentiment_asynchronous**.


## Account Simulation

- **Purpose:** Evaluate the single self-financing cross-sectional meta-filtered holdout account.
- **Settings:** [Shared portfolio settings](portfolio_management.ipynb); SPY benchmark; annual risk-free = 3%; benchmark Sharpe = 1; cash interest/borrow fees = 0.
- **Data:** Saved predictions, price calibration, market observations, account ledger, fills, and closed position episodes.
- **Findings:**
- **Decision:** Simulate once and use the same account state for every investment statistic.


In [ ]:
from src.backtesting.portfolio_management import strategy_settings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from src.backtesting.backtest_statistics import (
    ClassificationScores,
    Efficiency,
    GeneralCharacteristics,
    ImplementationShortfall,
    Performance,
    Runs,
)
from src.backtesting.portfolio_management import PortfolioSettings, run_final_backtest
from src.modeling.purged_validation import index_events
from src.preprocessing.market_data import ResearchPaths, save_frame

PROJECT_ROOT = Path.cwd().resolve().parents[2]
paths = ResearchPaths(PROJECT_ROOT, model_kind="sentiment")
STRATEGY = "sentiment_asynchronous"
settings = strategy_settings(STRATEGY)
annual_risk_free_rate = 0.03
annualized_benchmark_sharpe_ratio = 1.0
statistics, result = run_final_backtest(paths, settings, strategy=STRATEGY)

ledger = result["ledger"].set_index("timestamp")
closed = result["closed_trades"].copy()
start = GeneralCharacteristics.start(ledger.index)
end = GeneralCharacteristics.end(ledger.index)

daily = ledger[["aum"]].resample("D").last().ffill()
daily_ends = daily.index + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1)
daily.index = pd.DatetimeIndex([min(time, end) for time in daily_ends], name="timestamp")
daily["period_start"] = pd.Series([start, *daily.index[:-1]], index=daily.index)
previous_aum = daily["aum"].shift(1)
previous_aum.iloc[0] = settings.initial_aum
daily["net_return"] = daily["aum"] / previous_aum - 1.0

increments = ledger[["broker_fee", "slippage", "traded_value"]].diff()
increments.iloc[0] = ledger[["broker_fee", "slippage", "traded_value"]].iloc[0]
if result["exposures"].empty:
    position_values = pd.DataFrame({"_flat": 0.0}, index=ledger.index)
else:
    position_values = result["exposures"].pivot(
        index="timestamp", columns=["group", "symbol"] if "group" in result["exposures"] else "symbol", values="position_value"
    ).reindex(ledger.index).fillna(0.0)

equity = pd.concat([
    pd.Series([settings.initial_aum], index=pd.DatetimeIndex([start])),
    ledger["aum"],
])
if closed.empty:
    positions = event_end = trade_returns = None
else:
    trade_index = pd.DatetimeIndex(closed["event_start"])
    positions = pd.Series(closed["side"].to_numpy(), index=trade_index)
    event_end = pd.Series(pd.DatetimeIndex(closed["event_end"]), index=trade_index)
    trade_returns = pd.Series(closed["net_return"].to_numpy(), index=trade_index)

display(result["exclusions"])


## General Characteristics

- **Purpose:** Describe account span, capital, exposure, trade frequency, holding time, turnover, and benchmark relationship.
- **Settings:** Closed positions are flat-to-flat episodes; same-direction resizing stays in one episode; turnover uses 365.25-day years.
- **Data:** Calendar-day AUM, full-resolution symbol exposure, incremental traded notional, and SPY daily returns.
- **Findings:**
- **Decision:** Flat accounts report zero bet frequency and undefined direction or holding-period statistics.


In [ ]:
from src.backtesting.portfolio_management import benchmark_returns as load_benchmark_returns

account_returns = daily["net_return"].copy()
account_returns.index = account_returns.index.normalize()
spy_returns = load_benchmark_returns(paths, start, end)
spy_returns.index = spy_returns.index.normalize()

general_characteristics = pd.DataFrame([{
    "start": start,
    "end": end,
    "average_aum": GeneralCharacteristics.average_aum(daily["aum"]),
    "leverage": GeneralCharacteristics.leverage(position_values, ledger["aum"]),
    "maximum_dollar_position_size": GeneralCharacteristics.maximum_dollar_position_size(
        position_values
    ),
    "ratio_of_longs": GeneralCharacteristics.ratio_of_longs(positions)
    if positions is not None else np.nan,
    "frequency_of_bets": GeneralCharacteristics.frequency_of_bets(
        positions, event_end, time_range=(start, end)
    ) if positions is not None else 0.0,
    "average_holding_period": GeneralCharacteristics.average_holding_period(
        positions, event_end
    ) if positions is not None else np.nan,
    "annualized_turnover": GeneralCharacteristics.annualized_turnover(
        increments["traded_value"], daily["aum"], time_range=(start, end)
    ),
    "correlation_to_underlying": GeneralCharacteristics.correlation_to_underlying(
        account_returns, spy_returns
    ),
}], index=pd.Index(["final_account"], name="entity"))
display(general_characteristics)


## Performance

- **Purpose:** Report account PnL, elapsed-time annualized return, and closed-position outcomes.
- **Settings:** Annualize over elapsed 365.25-day years; attribute long/short PnL by each episode’s opening side.
- **Data:** Final account AUM and reconciled closed-position PnL and returns.
- **Findings:**
- **Decision:** Dollar PnL is not inferred by summing overlapping event returns.


In [ ]:
net_dollar_performance = pd.Series([
    ledger["aum"].iloc[-1] - settings.initial_aum
])
performance = pd.DataFrame([{
    "pnl": Performance.pnl(net_dollar_performance),
    "pnl_from_long_positions": Performance.pnl_from_long_positions(
        closed["net_pnl"], closed["side"]
    ) if not closed.empty else 0.0,
    "pnl_from_short_positions": Performance.pnl_from_short_positions(
        closed["net_pnl"], closed["side"]
    ) if not closed.empty else 0.0,
    "annualized_rate_of_return": Performance.annualized_rate_of_return(
        daily["net_return"], start=start, end=end
    ),
    "hit_ratio": Performance.hit_ratio(trade_returns)
    if trade_returns is not None else np.nan,
    "average_return_from_hits": Performance.average_return_from_hits(trade_returns)
    if trade_returns is not None else np.nan,
    "average_return_from_misses": Performance.average_return_from_misses(trade_returns)
    if trade_returns is not None else np.nan,
}], index=pd.Index(["final_account"], name="entity"))
np.testing.assert_allclose(closed["net_pnl"].sum(), performance.loc["final_account", "pnl"])
display(performance)


## Runs

- **Purpose:** Measure return and timing concentration together with drawdown depth and duration.
- **Settings:** HHI uses closed-position returns and entry months; episode percentiles use 95%.
- **Data:** The equity path includes initial capital and every retained account observation.
- **Findings:**
- **Decision:** Undefined concentration and no-drawdown percentiles remain NaN.


In [ ]:
runs = pd.DataFrame([{
    "hhi_positive_returns": Runs.hhi_positive_returns(trade_returns)
    if trade_returns is not None else np.nan,
    "hhi_negative_returns": Runs.hhi_negative_returns(trade_returns)
    if trade_returns is not None else np.nan,
    "hhi_time_between_bets": Runs.hhi_time_between_bets(trade_returns)
    if trade_returns is not None else np.nan,
    "percentile_drawdown": Runs.percentile_drawdown(equity),
    "percentile_time_under_water": Runs.percentile_time_under_water(equity),
}], index=pd.Index(["final_account"], name="entity"))
display(runs)


## Implementation Shortfall

- **Purpose:** Relate broker fees, slippage, and net performance to actual traded notional.
- **Settings:** Cumulative ledger counters are differenced before aggregation.
- **Data:** All entries, resizing trades, replacements, reductions, and final liquidations.
- **Findings:**
- **Decision:** Dollar performance is net of broker fees and adverse execution slippage.


In [ ]:
implementation_shortfall = pd.DataFrame([{
    "broker_fees_per_turnover": ImplementationShortfall.broker_fees_per_turnover(
        increments["broker_fee"], increments["traded_value"]
    ),
    "average_slippage_per_turnover": ImplementationShortfall.average_slippage_per_turnover(
        increments["slippage"], increments["traded_value"]
    ),
    "dollar_performance_per_turnover": ImplementationShortfall.dollar_performance_per_turnover(
        net_dollar_performance, increments["traded_value"]
    ),
    "return_on_execution_costs": ImplementationShortfall.return_on_execution_costs(
        net_dollar_performance,
        increments["broker_fee"] + increments["slippage"],
    ),
}], index=pd.Index(["final_account"], name="entity"))
display(implementation_shortfall)


## Efficiency

- **Purpose:** Evaluate return per unit of variability and the probability that annualized Sharpe exceeds one.
- **Settings:** Calendar-day observations, 365.25 annualization, and 3% effective annual risk-free return over actual intervals.
- **Data:** Full-account returns; the first period includes opening costs relative to initial capital.
- **Findings:**
- **Decision:** Cash earns no interest in the simulated account, while the risk-free rate remains the evaluation benchmark.


In [ ]:
efficiency = pd.DataFrame.from_dict({
    "final_account": Efficiency.portfolio_statistics(
        daily["net_return"],
        daily["period_start"],
        annual_risk_free_rate=annual_risk_free_rate,
        periods_per_year=365.25,
        annualized_benchmark_sharpe_ratio=annualized_benchmark_sharpe_ratio,
    )
}, orient="index").rename(columns={"annualized_sharpe": "annualized_sharpe_ratio"})
efficiency.index.name = "entity"
display(efficiency)


## Classification Scores

- **Purpose:** Evaluate the frozen primary direction and meta action classifiers on holdout events.
- **Settings:** Stored sample weights, positive label 1, and complete probability matrices.
- **Data:** Primary labels {-1, 1} and meta labels {0, 1} from the saved prediction artifact.
- **Findings:**
- **Decision:** Report accuracy, precision, recall, F1, and negative log loss for both models.


In [ ]:
predictions = index_events(pd.read_parquet(paths.artifacts / "meta_predictions.parquet"))
holdout = predictions.loc[predictions["partition"].eq("holdout")]
classification_inputs = {
    "primary": {
        "y_true": holdout["direction_label"],
        "y_pred": holdout["primary_side"],
        "probabilities": np.column_stack([
            1.0 - holdout["primary_probability"], holdout["primary_probability"]
        ]),
        "labels": [-1, 1],
    },
    "meta": {
        "y_true": holdout["meta_label"],
        "y_pred": holdout["meta_action"],
        "probabilities": np.column_stack([
            1.0 - holdout["meta_probability"], holdout["meta_probability"]
        ]),
        "labels": [0, 1],
    },
}
classification_columns = {}
for model, inputs in classification_inputs.items():
    classification_columns[model] = {
        "accuracy": ClassificationScores.accuracy(
            inputs["y_true"], inputs["y_pred"], sample_weight=holdout["sample_weight"]
        ),
        "precision": ClassificationScores.precision(
            inputs["y_true"], inputs["y_pred"], sample_weight=holdout["sample_weight"]
        ),
        "recall": ClassificationScores.recall(
            inputs["y_true"], inputs["y_pred"], sample_weight=holdout["sample_weight"]
        ),
        "f1_score": ClassificationScores.f1_score(
            inputs["y_true"], inputs["y_pred"], sample_weight=holdout["sample_weight"]
        ),
        "negative_log_loss": ClassificationScores.negative_log_loss(
            inputs["y_true"], inputs["probabilities"], labels=inputs["labels"],
            sample_weight=holdout["sample_weight"],
        ),
    }
classification_scores = pd.DataFrame(classification_columns)
display(classification_scores)


## Final Result

- **Purpose:** Persist strategy-specific statistics and account curves containing every statistics family shown above.
- **Settings:** Numeric values and UTC timestamps use separate typed columns.
- **Data:** Account, benchmark, exposure, classification, assumption, and coverage results.
- **Findings:**
- **Decision:** Recomputing the artifact requires rerunning the final account simulation and this ordered review.


In [ ]:
required_sections = {
    "general_characteristics",
    "performance",
    "runs",
    "implementation_shortfall",
    "efficiency",
    "classification_scores",
}
if not required_sections.issubset(set(statistics["section"])):
    raise ValueError("Final statistics are missing a required evaluation family")
result_path = PROJECT_ROOT / "data/backtest_results/sentiment_asynchronous/backtest_statistics.parquet"
save_frame(statistics, result_path)
display(statistics)
print(result_path)
